# Prospect Agent — Researcher → Copywriter Pipeline
## Research → Copy → Score — Outreach Automation Pipeline
⏱ ~12 min

**Agentic pipelines** chain specialist LLMs so that each one does only what it's best at — one gathers facts, another writes.
By the end of this session you will understand *why* role separation matters, *how* typed state flows between nodes in LangGraph,
and *how* to build a production-grade two-agent pipeline that searches the web and drafts personalized outreach.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — Linear pipelines vs ReAct loops; role separation |
| 2 | **Typed State** — `TypedDict` vs `MessagesState`; Pydantic models |
| 3 | **Researcher Node** — DuckDuckGo search + LLM fact extraction |
| 4 | **Copywriter Node** — Structured output with `with_structured_output` |
| 5 | **Graph Assembly** — Wiring the linear pipeline with LangGraph |
| 6 | **Batch Processing** — Running the pipeline over multiple prospects |
| 7 | **Debugging & Inspection** — Tracing state through each node |
| ★ | **Extensions** — QA gate, confidence filter, tone variants |

---

### Prerequisites
- Python 3.10+ with a `.venv` that has `requirements.txt` installed
- An `OPENAI_API_KEY` set in `.env` (or Colab Secrets)
- Internet access (the researcher node calls DuckDuckGo)

### Key References
> Yao, S., Zhao, J., et al. (2023). *ReAct: Synergizing Reasoning and Acting in Language Models.* ICLR 2023. https://arxiv.org/abs/2210.03629
>
> Wei, J., Wang, X., et al. (2022). *Chain-of-Thought Prompting Elicits Reasoning in Large Language Models.* NeurIPS 2022. https://arxiv.org/abs/2201.11903
>
> LangGraph documentation: https://langchain-ai.github.io/langgraph/
>
> LangChain structured output: https://python.langchain.com/docs/how_to/structured_output/

In [ ]:
# Install dependencies — runs only on Google Colab.
# Local users: your .venv already has everything from requirements.txt.
import sys


def _in_colab():
    try:
        import google.colab

        return True
    except ImportError:
        return False


if _in_colab():
    import subprocess

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "langchain",
            "langchain-openai",
            "langchain-community",
            "langgraph",
            "duckduckgo-search",
            "pydantic",
            "python-dotenv",
            "pandas",
        ]
    )
    print("Colab install complete.")
else:
    print("Local environment detected — skipping install.")

In [ ]:
import os
from typing import TypedDict

from pydantic import BaseModel, Field

# ── API key ───────────────────────────────────────────────────────────────────
# Colab: set in Secrets panel (left sidebar key icon)
# Local: create a .env file containing  OPENAI_API_KEY=sk-...
try:
    from google.colab import userdata

    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv

    load_dotenv()

# ── Sanity check ──────────────────────────────────────────────────────────────
key = os.environ.get("OPENAI_API_KEY", "")
assert key.startswith("sk-"), (
    "OPENAI_API_KEY missing or invalid. "
    "Local: add to .env  |  Colab: Secrets panel → 'OPENAI_API_KEY'"
)
print(f"OPENAI_API_KEY present: True (starts with sk-)")

---

## Part 1 — Concepts: Linear Pipelines vs ReAct Loops

---

### Two Fundamental Agentic Patterns

| Pattern | Structure | Best for |
|---------|-----------|----------|
| **ReAct Loop** (Reason + Act) | Single agent loops: Thought → Action → Observation → Thought … | Open-ended tasks where the agent decides what steps to take |
| **Linear Pipeline** | Fixed sequence of specialist nodes | Tasks with a known, deterministic stage order |
| **Conditional Graph** | Nodes with branching edges | Tasks that need routing (e.g., route by intent) |
| **Multi-Agent Graph** | Multiple pipelines sharing one state | Complex tasks that need parallel sub-tasks |

This workshop covers the **Linear Pipeline** — the simplest multi-agent pattern and the right choice when you know the exact stages in advance.

---

### Why Role Separation Matters

A single LLM asked to "research Jensen Huang and write a personalized outreach message" faces two conflicting objectives:
- **Research** requires precision: find verifiable facts, do not fabricate
- **Copywriting** requires creativity: be warm, specific, conversational

Splitting the work lets each LLM be given a tightly scoped system prompt optimized for its task.
The researcher can even use a different (often cheaper or slower) model than the copywriter.

---

### Architecture: Prospect Agent Flow

```
INPUT
  prospect = {first_name, last_name, company, position}
       │
       ▼
┌──────────────────────────────────────┐
│          RESEARCHER NODE             │
│                                      │
│  1. Build search query               │
│     "{name} {company} {position}"    │
│                                      │
│  2. Call DuckDuckGo Search           │
│     → raw web snippets               │
│                                      │
│  3. LLM extracts 2-3 key facts       │
│     (researcher model, temp=0)       │
│                                      │
│  State update:                       │
│    state["research"] = ResearchOutput│
└──────────────────┬───────────────────┘
                   │
                   ▼
┌──────────────────────────────────────┐
│          COPYWRITER NODE             │
│                                      │
│  Reads: state["prospect"]            │
│         state["research"].facts      │
│                                      │
│  LLM drafts ≤300 char outreach msg  │
│  with_structured_output → typed obj  │
│                                      │
│  State update:                       │
│    state["output"] = OutreachOutput  │
└──────────────────┬───────────────────┘
                   │
                   ▼
OUTPUT
  output = {generated_message, confidence}
```

> **Source**: Pipeline pattern adapted from Yao et al. (2023) — the ReAct paper from Google Brain — which established the principle of separating reasoning and acting steps in language agent design.

---

## Part 2 — Typed State: TypedDict vs MessagesState

---

### The State Is the Contract Between Nodes

LangGraph passes one state object through all nodes. Every node reads from it and returns a dict of *only the fields it updates*. LangGraph merges that dict back into state.

```python
# Node pattern — always the same shape:
def my_node(state: AgentState) -> dict:
    # read from state
    thing = state["some_field"]
    # compute
    result = do_something(thing)
    # return ONLY the fields this node updates
    return {"updated_field": result}
```

### MessagesState vs Custom TypedDict

| | `MessagesState` | Custom `TypedDict` |
|-|-----------------|--------------------|
| **Fields** | Single `messages: list[BaseMessage]` | You define each field explicitly |
| **Best for** | Conversational agents, chat history | Structured pipelines with named stages |
| **Inspection** | Print message list | Access `state["research"]`, `state["output"]` by name |
| **Pydantic integration** | Manual parsing | Fields can be Pydantic models with validation |
| **Typical use** | Chatbot, Q&A | Research pipeline, multi-stage processing |

**Rule of thumb**: use `MessagesState` when your agent is conversational. Use a custom `TypedDict` when your pipeline has distinct, named stages.

---

### Pydantic for Structured LLM Output

The copywriter node uses `llm.with_structured_output(OutreachOutput)` — this tells the LLM to return JSON that matches the Pydantic schema. Benefits:
- **Validation**: `confidence` must be 0.0–1.0, guaranteed
- **Type safety**: downstream code accesses `result.generated_message`, not `result["generated_message"]`
- **Clear contracts**: the schema documents what the LLM is expected to produce

> Wei et al. (2022) showed that structured prompting dramatically improves reliability — `with_structured_output` is the production implementation of that insight.

In [ ]:
# ─── 2-A: Define state and data models ────────────────────────────────────────
# Three Pydantic models carry typed data through the pipeline.
# AgentState is the TypedDict that LangGraph manages as its state object.


class ProspectMetadata(BaseModel):
    """Input: who we're researching."""

    first_name: str
    last_name: str
    company: str
    position: str


class ResearchOutput(BaseModel):
    """Filled by researcher_node."""

    search_query: str
    search_results: str  # 2-3 extracted fact sentences


class OutreachOutput(BaseModel):
    """Filled by copywriter_node — this is the final pipeline output."""

    generated_message: str = Field(description="Personalized outreach message (<=300 chars)")
    confidence: float = Field(ge=0.0, le=1.0, description="Confidence score 0-1")


class AgentState(TypedDict):
    """The shared state object passed through every node."""

    prospect: ProspectMetadata  # set at startup, never changed
    research: ResearchOutput | None  # filled by researcher_node
    output: OutreachOutput | None  # filled by copywriter_node


# Demonstrate state initialization
example_state: AgentState = {
    "prospect": ProspectMetadata(
        first_name="Jensen",
        last_name="Huang",
        company="NVIDIA",
        position="CEO",
    ),
    "research": None,  # not yet filled
    "output": None,  # not yet filled
}

print("Initial state:")
print(f"  prospect = {example_state['prospect']}")
print(f"  research = {example_state['research']}  (filled by researcher_node)")
print(f"  output   = {example_state['output']}   (filled by copywriter_node)")

In [ ]:
# ─── 2-B: Demonstrate Pydantic validation ─────────────────────────────────────
# Show what happens when structured output constraints are violated.
# This is why we use Pydantic rather than plain dicts — validation is automatic.

from pydantic import ValidationError

# Valid output — confidence is 0.0-1.0
valid = OutreachOutput(
    generated_message="Hi Jensen, I saw NVIDIA's H100 shipments just broke records — impressive scale.",
    confidence=0.85,
)
print(f"Valid output: confidence={valid.confidence}")
print(f"Message length: {len(valid.generated_message)} chars (<= 300 required)")
print()

# Invalid output — confidence out of range
try:
    invalid = OutreachOutput(
        generated_message="Hello!",
        confidence=1.5,  # violates ge=0.0, le=1.0
    )
except ValidationError as e:
    print("Pydantic caught invalid confidence:")
    print(f"  {e.errors()[0]['msg']}")

---

## Part 3 — Researcher Node: Web Search + Fact Extraction

---

### What the Researcher Node Does

1. Reads `state["prospect"]` to build a targeted search query
2. Calls `DuckDuckGoSearchResults` to fetch up to 5 web snippets
3. Passes the raw snippets to an LLM with a strict system prompt: extract 2–3 verifiable facts, nothing fabricated
4. Returns `{"research": ResearchOutput(...)}` — only the field it owns

### Why a Dedicated Search-Extract Pattern?

Directly feeding raw web snippets to the copywriter would:
- Waste tokens (snippets contain ads, navigation text, boilerplate)
- Risk hallucination (copywriter might "remember" facts not in the snippets)
- Mix concerns (research and writing in one prompt reduces quality for both)

The researcher acts as a **fact distillation layer** — the copywriter only sees clean, attributed sentences.

### Search Tool Comparison

| Tool | Cost | Rate limit | Best for |
|------|------|-----------|----------|
| `DuckDuckGoSearchResults` | Free | Aggressive (throttled) | Development, low volume |
| `BraveSearch` | Free tier / paid | Generous | Production, higher volume |
| `TavilySearchResults` | Paid | Generous | Best quality, LLM-optimized |
| `GoogleSerperAPIWrapper` | Paid | Generous | Google results |

This workshop uses DuckDuckGo (no API key required). The production `main.py` supports Brave as an alternative via `PREFERRED_SEARCH_PROVIDER=brave`.

In [ ]:
# ─── 3-A: Build and test the search tool standalone ──────────────────────────
# Always test individual components before wiring them into a graph.

from langchain_community.tools import DuckDuckGoSearchResults
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI

_search = DuckDuckGoSearchResults(max_results=5)
_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Test the search tool directly
test_prospect = ProspectMetadata(
    first_name="Jensen",
    last_name="Huang",
    company="NVIDIA",
    position="CEO",
)

query = f"{test_prospect.first_name} {test_prospect.last_name} {test_prospect.company} {test_prospect.position}"
print(f"Search query: '{query}'")
print()

raw_results = _search.run(query)
print("Raw DuckDuckGo results (first 600 chars):")
print(raw_results[:600])
print("...")

In [ ]:
# ─── 3-B: LLM fact extraction from raw search results ────────────────────────
# The researcher LLM distills noisy web snippets into 2-3 clean facts.
# temperature=0 → deterministic, no creative liberties.

fact_extraction_system = SystemMessage(
    "Extract 2-3 factual, verifiable sentences about this person and company "
    "from the search results. Be concise. Do not fabricate or infer — "
    "only include what is explicitly stated in the search results."
)

user_msg = HumanMessage(
    f"Prospect: {test_prospect.first_name} {test_prospect.last_name}, "
    f"{test_prospect.position} at {test_prospect.company}\n\n"
    f"Search results:\n{raw_results[:2000]}"
)

fact_response = _llm.invoke([fact_extraction_system, user_msg])

print("Extracted facts:")
print(fact_response.content)

In [ ]:
# ─── 3-C: Define the researcher_node function ─────────────────────────────────
# This is the exact function that will be registered as a LangGraph node.
# Note: it receives AgentState and returns a dict of only the fields it updates.


def researcher_node(state: AgentState) -> dict:
    """Search the web and extract 2-3 facts about the prospect."""
    p = state["prospect"]
    query = f"{p.first_name} {p.last_name} {p.company} {p.position}"
    print(f"  [researcher] searching: {query}")

    raw = _search.run(query)

    resp = _llm.invoke(
        [
            SystemMessage(
                "Extract 2-3 factual, verifiable sentences about this person and company "
                "from the search results. Be concise. Only include what is explicitly stated."
            ),
            HumanMessage(
                f"Prospect: {p.first_name} {p.last_name}, {p.position} at {p.company}\n\n"
                f"Search results:\n{raw[:2000]}"
            ),
        ]
    )

    # Return ONLY the fields this node is responsible for
    return {"research": ResearchOutput(search_query=query, search_results=resp.content)}


print("researcher_node defined.")
print("Signature: (state: AgentState) -> dict with key 'research'")

---

## Part 4 — Copywriter Node: Structured Output

---

### What `with_structured_output` Does

Normally, `llm.invoke(messages)` returns an `AIMessage` with a plain string `.content`. When you call `llm.with_structured_output(MyModel)`, LangChain:

1. Converts `MyModel` into a JSON schema
2. Passes it to the LLM as a function/tool definition
3. Forces the LLM to respond with valid JSON matching that schema
4. Parses the JSON and returns a validated `MyModel` instance

```python
# Without structured output:
result = llm.invoke(messages)             # → AIMessage(content="Here is my message...")
text = result.content                      # plain string, need to parse manually

# With structured output:
structured_llm = llm.with_structured_output(OutreachOutput)
result = structured_llm.invoke(messages)  # → OutreachOutput(generated_message=..., confidence=...)
text = result.generated_message           # typed field, validated
```

### Copywriter System Prompt Design

The copywriter prompt must:
- **Constrain length**: "<= 300 characters" — specific, measurable
- **Require specificity**: "reference a concrete fact" — prevents generic messages
- **Prohibit pitching**: "do NOT pitch services" — keeps it conversational
- **Set tone**: "friendly, genuine" — prevents corporate-speak

In [ ]:
# ─── 4-A: Demonstrate with_structured_output ──────────────────────────────────
# Run the copywriter logic standalone (not inside the graph) so you can
# see the typed output object before connecting nodes.

# Simulate what researcher_node would have put in state
fake_research = ResearchOutput(
    search_query="Jensen Huang NVIDIA CEO",
    search_results=(
        "Jensen Huang co-founded NVIDIA in 1993 and has served as CEO ever since. "
        "Under his leadership, NVIDIA's market cap surpassed $3 trillion in 2024. "
        "Huang is known for his leather jacket, a signature look at every NVIDIA keynote."
    ),
)

structured_llm = _llm.with_structured_output(OutreachOutput)

copywriter_result = structured_llm.invoke(
    [
        SystemMessage(
            "You are an expert outreach copywriter. Write a friendly, specific 1-sentence message "
            "(<=300 chars) that references a concrete fact about the prospect or their company. "
            "Do NOT pitch any services. Aim to start a genuine conversation."
        ),
        HumanMessage(
            f"Prospect: {test_prospect.first_name} {test_prospect.last_name}, "
            f"{test_prospect.position} at {test_prospect.company}\n\n"
            f"Research facts:\n{fake_research.search_results}"
        ),
    ]
)

print(f"Type: {type(copywriter_result).__name__}")
print(f"Message: {copywriter_result.generated_message}")
print(f"Length:  {len(copywriter_result.generated_message)} chars")
print(f"Confidence: {copywriter_result.confidence}")

In [ ]:
# ─── 4-B: Define the copywriter_node function ────────────────────────────────
# The copywriter reads research from state (filled by researcher_node) and
# returns a typed OutreachOutput object.


def copywriter_node(state: AgentState) -> dict:
    """Draft a personalized outreach message using the research facts."""
    p = state["prospect"]
    facts = state["research"].search_results
    print(f"  [copywriter] drafting for {p.first_name} {p.last_name}")

    structured_llm = _llm.with_structured_output(OutreachOutput)

    result = structured_llm.invoke(
        [
            SystemMessage(
                "You are an expert outreach copywriter. Write a friendly, specific 1-sentence message "
                "(<=300 chars) that references a concrete fact about the prospect or their company. "
                "Do NOT pitch any services. Aim to start a genuine conversation."
            ),
            HumanMessage(
                f"Prospect: {p.first_name} {p.last_name}, {p.position} at {p.company}\n\n"
                f"Research facts:\n{facts}"
            ),
        ]
    )

    # Return ONLY the field this node is responsible for
    return {"output": result}


print("copywriter_node defined.")
print("Signature: (state: AgentState) -> dict with key 'output'")

---

## Part 5 — Graph Assembly: Wiring the Linear Pipeline

---

### LangGraph Graph API Primer

LangGraph builds a directed graph where:
- **Nodes** are Python functions that transform state
- **Edges** define the execution order
- **`START`** and **`END`** are special sentinel nodes

```python
from langgraph.graph import StateGraph, START, END

graph = StateGraph(AgentState)          # declare the state schema
graph.add_node("researcher", fn)        # register a node
graph.add_edge(START, "researcher")     # define execution order
graph.add_edge("researcher", "copywriter")
graph.add_edge("copywriter", END)
app = graph.compile()                   # compile into a runnable
result = app.invoke(initial_state)      # run it
```

### Edge Types

| Edge type | Method | Use when |
|-----------|--------|----------|
| **Direct** | `add_edge(a, b)` | Always go from a to b |
| **Conditional** | `add_conditional_edges(a, router_fn)` | Route based on state value |
| **Fan-out** | `add_edge(a, [b, c])` | Run b and c in parallel |

This pipeline uses only direct edges — the flow is always `researcher → copywriter`.

In [ ]:
# ─── 5-A: Assemble and compile the graph ──────────────────────────────────────

from langgraph.graph import END, START, StateGraph

graph = StateGraph(AgentState)

# Register nodes
graph.add_node("researcher", researcher_node)
graph.add_node("copywriter", copywriter_node)

# Define execution order: START → researcher → copywriter → END
graph.add_edge(START, "researcher")
graph.add_edge("researcher", "copywriter")
graph.add_edge("copywriter", END)

# Compile into a runnable
app = graph.compile()

print("Graph compiled successfully.")
print("Execution order: START → researcher → copywriter → END")

In [ ]:
# ─── 5-B: Run the full pipeline ───────────────────────────────────────────────
# Use a well-known public figure so DuckDuckGo will find real facts.

PROSPECT = ProspectMetadata(
    first_name="Jensen",
    last_name="Huang",
    company="NVIDIA",
    position="CEO",
)

print(
    f"Running pipeline for: {PROSPECT.first_name} {PROSPECT.last_name}, "
    f"{PROSPECT.position} at {PROSPECT.company}\n"
)

result = app.invoke({"prospect": PROSPECT, "research": None, "output": None})

print()
print("=" * 60)
print("PIPELINE RESULT")
print("=" * 60)
print()
print("[Research Facts]")
print(result["research"].search_results)
print()
print("[Generated Outreach Message]")
print(result["output"].generated_message)
print()
print(f"Message length: {len(result['output'].generated_message)} chars")
print(f"Confidence: {result['output'].confidence:.2f}")

In [ ]:
# ─── 5-C: Inspect every state snapshot with stream() ─────────────────────────
# app.stream() yields the state after each node. Use this to debug
# exactly what each node received and what it produced.

print("Streaming state snapshots through the graph:\n")

for step_name, step_state in app.stream(
    {"prospect": PROSPECT, "research": None, "output": None}
):
    print(f"--- After node: {step_name} ---")
    for key, value in step_state.items():
        if value is None:
            print(f"  {key}: None")
        elif hasattr(value, "model_dump"):
            dumped = value.model_dump()
            for k, v in dumped.items():
                if isinstance(v, str) and len(v) > 100:
                    dumped[k] = v[:100] + "..."
            print(f"  {key}: {dumped}")
        else:
            print(f"  {key}: {str(value)[:100]}")
    print()

---

## Part 6 — Batch Processing: Multiple Prospects

---

### Why Batch?

The real use case is processing an entire LinkedIn connections CSV export — potentially hundreds of prospects. The production `main.py` reads from `data/Connections.csv` (skipping 3 header rows that LinkedIn adds) and filters by connection date.

For the workshop, we run a small batch of 3 prospects and print a summary table.

### Rate Limiting Considerations

DuckDuckGo aggressively rate-limits bots. The production code adds a configurable `SEARCH_WAIT_SECONDS` sleep before each search call. For workshop use, a 2-second pause is usually sufficient.

In [ ]:
# ─── 6-A: Run the pipeline over a batch of prospects ─────────────────────────
# Uses well-known public figures so DuckDuckGo will find real data.
# Adds a small sleep between calls to avoid rate limiting.

import time

BATCH_PROSPECTS = [
    ProspectMetadata(first_name="Satya", last_name="Nadella", company="Microsoft", position="CEO"),
    ProspectMetadata(first_name="Sam", last_name="Altman", company="OpenAI", position="CEO"),
    ProspectMetadata(first_name="Dario", last_name="Amodei", company="Anthropic", position="CEO"),
]

batch_results = []

for i, prospect in enumerate(BATCH_PROSPECTS):
    print(f"[{i+1}/{len(BATCH_PROSPECTS)}] Processing: {prospect.first_name} {prospect.last_name}")

    try:
        r = app.invoke({"prospect": prospect, "research": None, "output": None})
        batch_results.append({
            "name": f"{prospect.first_name} {prospect.last_name}",
            "company": prospect.company,
            "message": r["output"].generated_message,
            "confidence": r["output"].confidence,
            "chars": len(r["output"].generated_message),
        })
    except Exception as e:
        print(f"  ERROR: {e}")
        batch_results.append({
            "name": f"{prospect.first_name} {prospect.last_name}",
            "company": prospect.company,
            "message": f"ERROR: {e}",
            "confidence": 0.0,
            "chars": 0,
        })

    if i < len(BATCH_PROSPECTS) - 1:
        time.sleep(2)  # avoid DuckDuckGo rate limiting

print()
print("=" * 70)
print("BATCH RESULTS")
print("=" * 70)
print(f"{'Name':<22} {'Co':<12} {'Conf':>5} {'Chars':>5}  Message")
print("-" * 70)
for r in batch_results:
    msg_preview = r["message"][:35] + "..." if len(r["message"]) > 35 else r["message"]
    print(f"{r['name']:<22} {r['company']:<12} {r['confidence']:>5.2f} {r['chars']:>5}  {msg_preview}")

In [ ]:
# ─── 6-B: Export batch results to CSV ─────────────────────────────────────────
# In production, main.py reads a LinkedIn connections CSV and writes results
# to data/aug/Connections_aug_YYYYMMDDH.csv. This cell shows the pattern.

import pandas as pd
from pathlib import Path

if batch_results:
    df = pd.DataFrame(batch_results)
    output_path = Path("./batch_results.csv")
    df.to_csv(output_path, index=False)
    print(f"Results exported to: {output_path}")
    print()
    print(df[["name", "company", "confidence", "chars", "message"]].to_string(index=False))
else:
    print("No results to export.")

---

## Part 7 — Debugging and Inspection

---

### Common Failure Modes

| Failure | Symptom | Root cause | Fix |
|---------|---------|-----------|-----|
| **No facts found** | Research returns generic content | DuckDuckGo rate-limited or person too obscure | Add sleep; try Brave or Tavily |
| **Message too long** | `len(message) > 300` | Copywriter ignored length constraint | Add explicit length check in QA node |
| **Low confidence** | `confidence < 0.5` | Researcher couldn't find real facts | Log and skip; try different query |
| **Hallucinated facts** | Message mentions things not in research | Copywriter LLM drawing on training data | Strengthen system prompt; temp=0 |
| **Rate limit error** | `RatelimitException` from DuckDuckGo | Too many requests too fast | Increase `SEARCH_WAIT_SECONDS` |

### Debugging Checklist

1. **Print raw search results** — confirms the tool works and returns relevant snippets
2. **Print extracted facts** — confirms the researcher's distillation quality
3. **Check message length** — enforce <= 300 programmatically, not by trust
4. **Use `app.stream()`** — see state after each node (shown in 5-C)
5. **Check confidence score** — set a minimum threshold to filter low-quality outputs

In [ ]:
# ─── 7-A: Diagnostic — run nodes manually to inspect intermediate state ───────
# Instead of running the full compiled graph, call each node function directly.
# This lets you inspect the state between nodes without adding print statements.


def run_with_diagnostics(prospect: ProspectMetadata) -> dict:
    """Run the pipeline step-by-step and print diagnostics at each stage."""

    print(f"\n{'='*60}")
    print(f"DIAGNOSTIC RUN: {prospect.first_name} {prospect.last_name}")
    print(f"{'='*60}")

    initial_state = {"prospect": prospect, "research": None, "output": None}

    # Step 1: run researcher manually
    print("\n[STEP 1: researcher_node]")
    research_update = researcher_node(initial_state)
    print(f"  Search query: {research_update['research'].search_query}")
    print(f"  Extracted facts:")
    print(f"    {research_update['research'].search_results}")

    # Merge update into state (simulating LangGraph's state merge)
    state_after_research = {**initial_state, **research_update}

    # Step 2: run copywriter manually
    print("\n[STEP 2: copywriter_node]")
    copywriter_update = copywriter_node(state_after_research)
    msg = copywriter_update["output"].generated_message
    conf = copywriter_update["output"].confidence
    print(f"  Message ({len(msg)} chars): {msg}")
    print(f"  Confidence: {conf:.2f}")
    print(f"  Length check: {'PASS' if len(msg) <= 300 else 'FAIL — over 300 chars'}")

    return {**state_after_research, **copywriter_update}


debug_result = run_with_diagnostics(
    ProspectMetadata(first_name="Jensen", last_name="Huang", company="NVIDIA", position="CEO")
)

In [ ]:
# ─── 7-B: Confidence filter — skip low-quality outputs ────────────────────────
# In production, flag or skip results where the copywriter was not confident.
# This usually means the researcher didn't find useful facts.

CONFIDENCE_THRESHOLD = 0.6

print(f"Reviewing batch results against threshold: {CONFIDENCE_THRESHOLD}\n")

if batch_results:
    for r in batch_results:
        conf = r.get("confidence", 0.0)
        status = "PASS" if conf >= CONFIDENCE_THRESHOLD else "FLAGGED"
        print(f"  [{status}] {r['name']}: confidence={conf:.2f}")
        if status == "FLAGGED":
            print(f"         Reason: confidence below {CONFIDENCE_THRESHOLD} — review research quality")
else:
    print("Run the batch cell (6-A) first to populate batch_results.")

---

## Exercises

---

Work through these in order. Each builds on the previous.

In [ ]:
# ── Exercise 1 — Change the Prospect ─────────────────────────────────────────
# Replace MY_PROSPECT with someone from your own network or a public figure.
# After running, evaluate:
#   - Did the researcher find useful, verifiable facts?
#   - Does the message reference a specific fact (not generic praise)?
#   - Would YOU respond to this message?

MY_PROSPECT = ProspectMetadata(
    first_name="TODO",   # Replace with a real first name
    last_name="TODO",    # Replace with a real last name
    company="TODO",      # Replace with their company
    position="TODO",     # Replace with their title
)

# TODO: uncomment and run after filling in MY_PROSPECT
# result = app.invoke({"prospect": MY_PROSPECT, "research": None, "output": None})
# print("Research:", result["research"].search_results)
# print("Message:", result["output"].generated_message)
# print("Confidence:", result["output"].confidence)

In [ ]:
# ── Exercise 2 — Add a QA Gate Node ──────────────────────────────────────────
# Add a third node: qa_node.
# It checks: (a) message is <= 300 chars, (b) confidence >= 0.6
# If either check fails, print a warning.
# Wire it: researcher → copywriter → qa → END


def qa_node(state: AgentState) -> dict:
    output = state["output"]
    # TODO: check len(output.generated_message) <= 300
    # TODO: check output.confidence >= 0.6
    # TODO: print PASS or WARN for each check
    # This node observes only — return empty dict (no state changes)
    return {}


# TODO: rebuild the graph with qa_node added
# graph_with_qa = StateGraph(AgentState)
# graph_with_qa.add_node("researcher", researcher_node)
# graph_with_qa.add_node("copywriter", copywriter_node)
# graph_with_qa.add_node("qa", qa_node)
# graph_with_qa.add_edge(START, "researcher")
# graph_with_qa.add_edge("researcher", "copywriter")
# graph_with_qa.add_edge("copywriter", "qa")
# graph_with_qa.add_edge("qa", END)
# app_with_qa = graph_with_qa.compile()
# app_with_qa.invoke({"prospect": PROSPECT, "research": None, "output": None})

print("TODO: implement qa_node and rebuild the graph")

In [ ]:
# ── Exercise 3 — Separate LLMs Per Node ──────────────────────────────────────
# The researcher and copywriter don't have to use the same model.
# The production main.py uses a deep-research model for the researcher
# and a faster model for the copywriter.
#
# Try:
#   researcher → gpt-4o-mini (fast, precise, temp=0)
#   copywriter → gpt-4o (higher quality writing, temp=0.3)
#
# Question: does output quality improve? At what cost (latency, tokens)?

# TODO: create two separate ChatOpenAI instances
# researcher_llm_ex3 = ChatOpenAI(model="gpt-4o-mini", temperature=0)
# copywriter_llm_ex3 = ChatOpenAI(model="gpt-4o", temperature=0.3)

# TODO: create node functions that use these specific LLMs
# def researcher_node_ex3(state): ...
# def copywriter_node_ex3(state): ...

# TODO: rebuild the graph and run on PROSPECT
print("TODO: create separate LLM instances and node functions for each node")

In [ ]:
# ── Exercise 4 — Batch Mode with Progress Table ───────────────────────────────
# Wrap app.invoke() in a loop over 5 different prospects.
# Print a formatted table: Name | Company | Confidence | Chars | Message preview
# Add sleep(2) between calls to avoid DuckDuckGo rate limiting.

MY_BATCH = [
    ProspectMetadata(first_name="TODO", last_name="TODO", company="TODO", position="TODO"),
    # Add 4 more prospects here...
]

# TODO: loop over MY_BATCH, invoke the pipeline for each
# TODO: collect results in a list of dicts with keys: name, company, confidence, chars, message
# TODO: print a formatted table (see 6-A for the table pattern)
print("TODO: fill in MY_BATCH with 5 prospects and run the loop")

---

## Part 8 ★ — Extensions

---

### Alternative Search Providers

DuckDuckGo has aggressive rate limits. For production use, swap in Brave or Tavily:

```python
# Brave Search (requires BRAVE_API_KEY)
from langchain_community.tools import BraveSearch
search = BraveSearch.from_api_key(api_key=os.getenv("BRAVE_API_KEY"), search_kwargs={"count": 5})

# Tavily (requires TAVILY_API_KEY — designed for LLM-friendly retrieval)
from langchain_community.tools.tavily_search import TavilySearchResults
search = TavilySearchResults(max_results=5)
```

The production `tools.py` supports both via `PREFERRED_SEARCH_PROVIDER=brave` env var.

### Conditional Routing: Skip Known Prospects

Add a `check_cache` node before the researcher. If the research is already cached, skip straight to the copywriter:

```python
def route_after_cache(state: AgentState) -> str:
    if state["research"] is not None:   # cache hit
        return "copywriter"
    return "researcher"                  # cache miss

graph.add_conditional_edges("check_cache", route_after_cache)
```

### LinkedIn CSV Integration

The production `main.py` reads a LinkedIn connections CSV export:
- Skips 3 header rows added by LinkedIn
- Filters by `Connected On` date range (`--since_date`, `--to_date`)
- Retries failed prospects with exponential backoff (up to 2 retries)
- Saves partial results on failure so work is not lost

```bash
python main.py --input data/Connections.csv \
               --output_suffix data/aug/Connections_aug \
               --since_date 2025-01-01 \
               --to_date 2025-06-01
```

In [ ]:
# ─── 8-A: Vary the copywriter tone ────────────────────────────────────────────
# The system prompt determines the entire personality of the output.
# Compare three tone variants on the same prospect + facts.

TONE_PROMPTS = {
    "formal": (
        "You are a professional business development executive. Write a concise, "
        "formal 1-sentence outreach message (<=300 chars) referencing a specific fact "
        "about the prospect. No sales pitch."
    ),
    "casual": (
        "You write like a friendly peer, not a salesperson. Write a casual 1-sentence "
        "message (<=300 chars) that feels like it's from a real person who did their homework. "
        "Mention something specific about the prospect."
    ),
    "question": (
        "Start the message with a genuine question about something specific the prospect "
        "has done or their company is working on (<=300 chars). No pitch."
    ),
}

# Use pre-fetched facts if available, else use a fallback
test_facts = (
    debug_result["research"].search_results
    if debug_result.get("research")
    else (
        "Jensen Huang co-founded NVIDIA in 1993. "
        "NVIDIA's market cap surpassed $3 trillion in 2024."
    )
)

print(f"Prospect: Jensen Huang, CEO at NVIDIA")
print(f"Facts: {test_facts[:150]}...\n")

for tone_name, tone_prompt in TONE_PROMPTS.items():
    structured_llm_tone = _llm.with_structured_output(OutreachOutput)
    tone_result = structured_llm_tone.invoke([
        SystemMessage(tone_prompt),
        HumanMessage(f"Prospect: Jensen Huang, CEO at NVIDIA\n\nFacts:\n{test_facts}"),
    ])
    print(f"[{tone_name.upper()}] ({len(tone_result.generated_message)} chars)")
    print(f"  {tone_result.generated_message}")
    print()

In [ ]:
# ─── 8-B: Drop-in Tavily search replacement ───────────────────────────────────
# If you have a TAVILY_API_KEY, this produces cleaner research results
# because Tavily is designed specifically for LLM-friendly web retrieval.

tavily_key = os.environ.get("TAVILY_API_KEY")

if tavily_key:
    try:
        from langchain_community.tools.tavily_search import TavilySearchResults

        _search_tavily = TavilySearchResults(max_results=5)
        test_query = "Jensen Huang NVIDIA CEO"
        tavily_raw = _search_tavily.invoke(test_query)

        print(f"Tavily results for '{test_query}':")
        for r in tavily_raw[:3]:
            print(f"  [{r.get('title', '?')}]")
            print(f"  {r.get('content', '')[:150]}...")
            print()
    except ImportError:
        print("Install: pip install tavily-python")
else:
    print("TAVILY_API_KEY not set — skipping.")
    print("Free key at https://tavily.com — add TAVILY_API_KEY to .env to enable")

---

## What's Next?

You now understand linear agentic pipelines, typed state, and structured LLM output. Here's where to go deeper:

### More nodes, more agents
- **Example 6 — Multi-Agent Graph** (`../6-multi-agent-graph/`): three specialist agents sharing one state graph — extends the two-node pattern here with parallel branches and more complex routing.

### Structured output in depth
- **Example 13 — Structured Output** (`../13-structured-output/`): deep dive into `with_structured_output` with nested schemas, enum fields, and validation error handling.

### Self-improving loops
- **Example 8 — New Idea Gen** (`../8-new-idea-gen/`): an LLM generates ideas, then critiques its own output in a loop — the next step after linear pipelines.

### RAG as the researcher's knowledge source
- **Example 1 — Basic Local RAG** (`../1-basic-local-rag/`): instead of web search, use a vector store as the researcher's data source — same node pattern, different retrieval mechanism.

### Further reading
- Yao, S., et al. (2023). *ReAct: Synergizing Reasoning and Acting in Language Models.* ICLR 2023. https://arxiv.org/abs/2210.03629
- Wei, J., et al. (2022). *Chain-of-Thought Prompting Elicits Reasoning in Large Language Models.* NeurIPS 2022. https://arxiv.org/abs/2201.11903
- LangGraph — Multi-agent systems: https://langchain-ai.github.io/langgraph/concepts/multi_agent/
- LangChain structured output guide: https://python.langchain.com/docs/how_to/structured_output/

---

*Workshop complete. The next step is example 6 — add a third specialist agent and see how state flows through a three-node graph.*

---

## Exercise Answer Key

Use this section as a reference **after** attempting the exercises yourself.
These are sample solutions — not the only correct answers.

### Exercise 1 — Change the Prospect

**What good output looks like:**
- The research facts cite something verifiable (a recent news item, a product launch, a public statement)
- The message references that specific fact by name — not just "your company is doing great work"
- The confidence score is >= 0.7 for public figures; it may be lower for less-covered people

**If you get generic or hallucinated facts:**
- Try a more senior person at a well-known company (easier for DuckDuckGo to find)
- Check `raw_results` in cell 3-A to see what the search returned before LLM extraction
- The researcher prompt says "only include what is explicitly stated" — if the LLM still fabricates, lower temperature to 0

**If DuckDuckGo returns nothing:**
- Add a longer `time.sleep(5)` before the search call
- Or switch to Tavily (cell 8-B) for more reliable results

In [ ]:
# ── Answer Key: Exercise 2 — QA Gate Node ─────────────────────────────────────

def qa_node_answer(state: AgentState) -> dict:
    """Quality assurance gate — checks output constraints without changing state."""
    output = state["output"]

    msg_len = len(output.generated_message)
    conf = output.confidence

    length_ok = msg_len <= 300
    conf_ok = conf >= 0.6

    print(f"  [qa] length: {msg_len} chars — {'PASS' if length_ok else 'WARN: over 300 chars'}")
    print(f"  [qa] confidence: {conf:.2f} — {'PASS' if conf_ok else 'WARN: below 0.6'}")

    # Observes only — return empty dict to leave state unchanged
    return {}


graph_with_qa = StateGraph(AgentState)
graph_with_qa.add_node("researcher", researcher_node)
graph_with_qa.add_node("copywriter", copywriter_node)
graph_with_qa.add_node("qa", qa_node_answer)
graph_with_qa.add_edge(START, "researcher")
graph_with_qa.add_edge("researcher", "copywriter")
graph_with_qa.add_edge("copywriter", "qa")
graph_with_qa.add_edge("qa", END)
app_with_qa = graph_with_qa.compile()

print("Graph with QA gate compiled.")
print("Execution order: START → researcher → copywriter → qa → END")

In [ ]:
# ── Answer Key: Exercise 3 — Separate LLMs ────────────────────────────────────

researcher_llm_ex3 = ChatOpenAI(model="gpt-4o-mini", temperature=0)  # fast, analytical
copywriter_llm_ex3 = ChatOpenAI(model="gpt-4o", temperature=0.3)     # higher quality writing


def researcher_node_ex3(state: AgentState) -> dict:
    p = state["prospect"]
    query = f"{p.first_name} {p.last_name} {p.company} {p.position}"
    print(f"  [researcher/gpt-4o-mini] searching: {query}")
    raw = _search.run(query)
    resp = researcher_llm_ex3.invoke([
        SystemMessage("Extract 2-3 factual, verifiable sentences. Only what is explicitly stated."),
        HumanMessage(f"{p.first_name} {p.last_name}, {p.position} at {p.company}\n\n{raw[:2000]}"),
    ])
    return {"research": ResearchOutput(search_query=query, search_results=resp.content)}


def copywriter_node_ex3(state: AgentState) -> dict:
    p = state["prospect"]
    print(f"  [copywriter/gpt-4o] drafting for {p.first_name} {p.last_name}")
    structured = copywriter_llm_ex3.with_structured_output(OutreachOutput)
    result = structured.invoke([
        SystemMessage(
            "Write a friendly, specific 1-sentence outreach message (<=300 chars) "
            "referencing a concrete fact. Do NOT pitch services."
        ),
        HumanMessage(
            f"Prospect: {p.first_name} {p.last_name}, {p.position} at {p.company}\n\n"
            f"Research:\n{state['research'].search_results}"
        ),
    ])
    return {"output": result}


graph_ex3 = StateGraph(AgentState)
graph_ex3.add_node("researcher", researcher_node_ex3)
graph_ex3.add_node("copywriter", copywriter_node_ex3)
graph_ex3.add_edge(START, "researcher")
graph_ex3.add_edge("researcher", "copywriter")
graph_ex3.add_edge("copywriter", END)
app_ex3 = graph_ex3.compile()

print("Split-LLM graph compiled.")
print("  researcher: gpt-4o-mini (fast, temp=0)")
print("  copywriter: gpt-4o (quality, temp=0.3)")

In [ ]:
# ── Answer Key: Exercise 4 — Batch Mode with Progress Table ───────────────────

SAMPLE_BATCH = [
    ProspectMetadata(first_name="Satya",  last_name="Nadella", company="Microsoft", position="CEO"),
    ProspectMetadata(first_name="Sam",    last_name="Altman",  company="OpenAI",    position="CEO"),
    ProspectMetadata(first_name="Dario",  last_name="Amodei",  company="Anthropic", position="CEO"),
    ProspectMetadata(first_name="Sundar", last_name="Pichai",  company="Google",    position="CEO"),
    ProspectMetadata(first_name="Tim",    last_name="Cook",    company="Apple",     position="CEO"),
]

print("Sample batch solution (uses pre-fetched batch_results if available):")
print()
print(f"{'#':<3} {'Name':<22} {'Company':<12} {'Conf':>5} {'Chars':>5}  Message preview")
print("-" * 75)

if batch_results:
    for i, r in enumerate(batch_results, 1):
        msg_preview = r.get("message", "")[:40]
        if len(r.get("message", "")) > 40:
            msg_preview += "..."
        print(
            f"{i:<3} {r['name']:<22} {r['company']:<12} "
            f"{r.get('confidence', 0):>5.2f} {r.get('chars', 0):>5}  {msg_preview}"
        )
else:
    print("  (run the batch cell in Part 6 first to populate results)")

print()
print("Note: SAMPLE_BATCH above shows the full 5-prospect list.")
print("To run the full batch: loop over SAMPLE_BATCH with time.sleep(2) between calls.")